In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
sales_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("cust_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("city", StringType(), True),
    StructField("region", StringType(), True),
    StructField("base_price", StringType(), True),
    StructField("discount_amount", StringType(), True),
    StructField("final_price", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("payment_mode", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("found_us_via", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("row_hash", StringType(), True)
])

In [0]:
source_path = "/Volumes/retail_project/bronze/sales_data_landing/"
checkpoint_path = "/Volumes/retail_project/bronze/sales_data_landing/_checkpoints/bronze/"
target_table = "retail_project.bronze.sales_raw"

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.inferColumnTypes", False)
    .option("header", True)
    .option("rescuedDataColumn", "_rescued_data")
    .schema(sales_schema)
    .load(source_path)
)

In [0]:
bronze_df = (
    df.withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("year", F.year(F.to_timestamp("event_ts")))
    .withColumn("month", F.month(F.to_timestamp("event_ts")))
)

In [0]:
save_df = (
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", True)
    .partitionBy("year", "month")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(target_table)
)

save_df.awaitTermination()